# APEX AI — Phase 5: LLM Integration (Google Colab Free GPU)

This notebook connects open-weight LLMs (like **Llama-3-8B-Instruct** or **Mistral-7B**) to the **APEX AI** Formula 1 Analytics Engine and Phase 4 Machine Learning models.

### System Architecture:
```
User Natural Language Query
       │
       ▼
Llama-3-8B on Free T4 GPU
  (Decides which tool to call)
       │
       ▼
APEX AI Tool Registry (tools.py)
  ├─ get_standings()
  ├─ driver_career_stats()
  ├─ consistency_ranking()
  └─ predict_race() -> XGBoost (91.5% Podium Acc)
       │
       ▼
LLM formats structured data into fluent, analytical English
```

---

## Step 1: Install Dependencies & Check GPU
Make sure your Colab Runtime is set to **T4 GPU** (Runtime → Change runtime type → T4 GPU).

In [ ]:
# Verify free T4 GPU
!nvidia-smi

# Install lightweight LLM inference stack
!pip install -q transformers accelerate bitsandbytes sentencepiece xgboost lightgbm pandas scikit-learn

## Step 2: Clone Repository or Download APEX AI Package

In [ ]:
# Clone the GitHub repository
!git clone https://github.com/bhxvxsh/yeah-def-my-nyc-project.git repo

import sys
import os
sys.path.insert(0, '/content/repo/APEX_AI')
sys.path.insert(0, '/content/repo/APEX_AI/05_Analytics')
sys.path.insert(0, '/content/repo/APEX_AI/04_Models')
os.chdir('/content/repo/APEX_AI')

# Verify tool imports
from tools import execute_tool, TOOL_SCHEMAS
print("Available APEX AI Tools:", [t['name'] for t in TOOL_SCHEMAS])

## Step 3: Load 4-bit Quantized Llama-3-8B / Mistral-7B (Fits comfortably in 16GB VRAM)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_id = "meta-llama/Meta-Llama-3-8B-Instruct"
# Or open alternative: "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    load_in_4bit=True
)

generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

## Step 4: Prompt Engine & Function Calling Orchestrator

In [ ]:
import json

SYSTEM_PROMPT = """You are APEX AI, a Formula 1 Chief Race Strategist and Performance Analyst.
You have access to structured analytical tools and machine-learning models trained on 2023-2025 F1 telemetry.

When answering the user:
1. If the user asks for standings, consistency, deltas, or race predictions, specify which tool to call in JSON format:
   ```json
   {"tool": "<tool_name>", "arguments": {<args>}}
   ```
2. When tool data is provided, synthesize the numbers into an expert, high-insight racing summary.

Available tools:
- get_standings(season: int) -> 2023, 2024, or 2025 championship table
- driver_career_stats(driver_id: str) -> multi-season wins, podiums, points (e.g. 'VER', 'HAM', 'NOR')
- consistency_ranking(top_n: int) -> most consistent drivers ranked by position stddev
- qualifying_race_delta(driver_id: str, season: int) -> positions gained from Saturday to Sunday
- predict_race(driver_id: str, grid_position: int) -> ML model race finish prediction with podium & points probabilities
"""

def ask_apex_agent(user_query: str):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    outputs = generator(prompt, max_new_tokens=256, do_sample=False)
    response = outputs[0]["generated_text"][len(prompt):].strip()
    
    # Check if LLM requested a tool call
    if "```json" in response and """tool""" in response:
        raw_json = response.split("```json")[1].split("```")[0].strip()
        call_info = json.loads(raw_json)
        tool_name = call_info["tool"]
        args = call_info.get("arguments", {})
        
        print(f"[Executing Tool]: {tool_name}({args})")
        tool_output = execute_tool(tool_name, args)
        
        # Pass tool result back to LLM for final synthesis
        messages.append({"role": "assistant", "content": response})
        messages.append({"role": "user", "content": f"Tool Execution Output:\n{tool_output}\nPlease synthesize this into a final response."})
        
        final_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        final_out = generator(final_prompt, max_new_tokens=300, do_sample=True, temperature=0.7)
        return final_out[0]["generated_text"][len(final_prompt):].strip()
    
    return response

## Step 5: Test the End-to-End LLM Agent

In [ ]:
query = "Predict what happens if Max Verstappen starts from P4 in Sunday's Grand Prix. What are his podium and points odds?"
print("User Query:", query)
print("-" * 70)
answer = ask_apex_agent(query)
print(answer)